In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 1 · [SETUP]  Install / upgrade all dependencies          ║
# ╚══════════════════════════════════════════════════════════════════╝
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"],
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("[STDERR]", result.stderr[-2000:])
    raise RuntimeError("pip install failed — see stderr above")
print("\n✓ All requirements installed successfully.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 2 · [VERIFY DATA]  Load, expand, engineer, split, save   ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # ensure agent/ is importable

import numpy as np
from agent.data.pipeline import DataPipeline
from agent.core.tee_logger import TeeLogger

logger = TeeLogger(master_log_dir="../master_log")
logger.info("=== Phase 1 · Data Verification Start ===")

pipe = DataPipeline(splits_dir="../data/splits")

logger.info("Loading and expanding CSV …")
df = pipe.load_and_expand("../data/raw/alpha_synuclein.csv")
logger.info(f"Expanded DataFrame shape: {df.shape}")

logger.info("Engineering features (this may take ~30 s for 3-mers) …")
X, y = pipe.build_features(df)
logger.info("Feature matrix built.")

logger.info("Splitting into train / val / test …")
pipe.stratified_split(X, y, train=0.70, val=0.15, test=0.15)

logger.info("Saving splits to disk …")
pipe.save_splits()

print(f"\nTotal samples      : {len(X)}")
print(f"Feature dimensions : {X.shape[1]}")
unique, counts = np.unique(y, return_counts=True)
print(f"Class distribution : {dict(zip(unique.tolist(), counts.tolist()))}")
logger.info("=== Phase 1 · Data Verification Complete ===")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 3 · [VERIFY WALL]  Seal and verify the mathematical wall ║
# ╚══════════════════════════════════════════════════════════════════╝
from agent.core.tee_logger import TeeLogger
logger = TeeLogger()

logger.info("Sealing test set …")
pipe.seal_test_set()

logger.info("Verifying wall integrity …")
wall_intact = pipe.verify_wall()

if wall_intact:
    logger.info("Mathematical wall is INTACT. Safe to proceed.")
else:
    logger.error("WALL BREACH — test set has been tampered with!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 4 · [LAUNCH]  Phase 3: Autonomous Agent                  ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from agent.core.orchestrator import AgentOrchestrator

# ── Choose your LLM — change this ONE line ──────────────────────────────────
# Local (no API key needed, requires Ollama running):
agent = AgentOrchestrator(model_name='local-qwen')

# Cloud alternatives (add key to ../.env first):
# agent = AgentOrchestrator(model_name='gemini-flash')   # GEMINI_API_KEY
# agent = AgentOrchestrator(model_name='groq-llama')     # GROQ_API_KEY
# agent = AgentOrchestrator(model_name='mistral-small')  # MISTRAL_API_KEY

print('Agent status:', agent.status())
print()
print('Starting autonomous research loop (max 200 experiments)...')
print('Press Kernel > Interrupt to stop gracefully.')
print()

# ── Launch ────────────────────────────────────────────────────────────────────
agent.run(max_experiments=200)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 5 · [MONITOR DASHBOARD]  Live auto-refreshing leaderboard ║
# ╚══════════════════════════════════════════════════════════════════╝
import json, time, os, threading
from pathlib import Path
from IPython.display import clear_output, display, HTML
import ipywidgets as widgets

_LB_PATH    = Path('../master_log/leaderboard.json')
_STATE_PATH = Path('../master_log/orchestrator_state.json')
_REFRESH_S  = 30   # seconds between auto-refresh

LABEL_NAMES = {0: 'No', 1: 'Low', 2: 'Medium', 3: 'High'}

def _gpu_info():
    try:
        import subprocess
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=3
        )
        if r.returncode == 0:
            used, total, util = r.stdout.strip().split(',')
            return f'GPU: {used.strip()} / {total.strip()} MB  |  {util.strip()}% util'
    except Exception:
        pass
    return 'GPU: info unavailable'

def _read_leaderboard():
    if not _LB_PATH.exists():
        return {}
    with open(_LB_PATH) as f:
        return json.load(f)

def _read_state():
    if not _STATE_PATH.exists():
        return {}
    with open(_STATE_PATH) as f:
        return json.load(f)

def _progress_bar(done, total=200, width=40):
    filled = int(width * done / max(total, 1))
    bar    = '#' * filled + '-' * (width - filled)
    pct    = 100 * done / max(total, 1)
    return f'[{bar}] {done}/{total} ({pct:.1f}%)'

def _render_dashboard(out_widget, max_experiments=200):
    lb    = _read_leaderboard()
    state = _read_state()

    exps   = lb.get('experiments', [])
    n_done = lb.get('total_runs', 0)
    best_f1= lb.get('best_val_f1_macro', 0.0)
    best_id= lb.get('best_experiment', 'none')
    fams   = lb.get('families_completed', [])
    model  = lb.get('agent_model_used', '?')
    updated= lb.get('last_updated', 'never')

    agent_status = state.get('status', 'unknown')

    lines = []
    lines.append(f'<h2 style="margin:0;color:#4fc3f7">Alpha-Synuclein Agent — Live Dashboard</h2>')
    lines.append(f'<small>Auto-refresh every {_REFRESH_S}s &nbsp;|&nbsp; Last update: {updated}</small><br/><br/>')

    # Status bar
    status_color = {'running': '#66bb6a', 'idle': '#bdbdbd',
                    'stopping': '#ffa726', 'unknown': '#ef5350'}.get(agent_status, '#bdbdbd')
    lines.append(
        f'<b>Agent:</b> <span style="color:{status_color}">&#9646; {agent_status.upper()}</span>'
        f'&nbsp;&nbsp;<b>Model:</b> {model}<br/>'
        f'<b>Progress:</b> {_progress_bar(n_done, max_experiments)}<br/>'
        f'<b>Best F1-macro:</b> <span style="font-size:1.2em;color:#ffb74d">{best_f1:.4f}</span>'
        f' ({best_id})&nbsp;&nbsp;'
        f'<b>{_gpu_info()}</b><br/><br/>'
    )

    # Top experiments table
    ranked = sorted(exps, key=lambda e: e.get('val_f1_macro', 0), reverse=True)[:10]
    if ranked:
        tbl  = '<table style="border-collapse:collapse;width:100%;font-size:0.85em">'
        tbl += '<tr style="background:#263238">'
        for hdr in ['Rank', 'Exp ID', 'Architecture', 'Family', 'F1-macro', 'Accuracy', 'Train(s)', 'Status']:
            tbl += f'<th style="padding:4px 8px;text-align:left;color:#b0bec5">{hdr}</th>'
        tbl += '</tr>'
        for rank, exp in enumerate(ranked, 1):
            f1    = exp.get('val_f1_macro', 0)
            acc   = exp.get('val_accuracy', 0)
            t     = exp.get('train_time_seconds', 0)
            st    = exp.get('status', '?')
            bg    = '#1b2631' if rank % 2 == 0 else '#1a252f'
            f1_c  = '#66bb6a' if f1 >= 0.6 else ('#ffa726' if f1 >= 0.4 else '#ef5350')
            st_c  = '#66bb6a' if st == 'success' else '#ef5350'
            tbl  += (
                f'<tr style="background:{bg}">'
                f'<td style="padding:4px 8px;color:#90a4ae">{rank}</td>'
                f'<td style="padding:4px 8px;color:#b0bec5">{exp.get("exp_id","?")}</td>'
                f'<td style="padding:4px 8px;color:#e0e0e0">{exp.get("architecture","?")[:24]}</td>'
                f'<td style="padding:4px 8px;color:#80cbc4">{exp.get("architecture_family","?")[:16]}</td>'
                f'<td style="padding:4px 8px;font-weight:bold;color:{f1_c}">{f1:.4f}</td>'
                f'<td style="padding:4px 8px;color:#90a4ae">{acc:.4f}</td>'
                f'<td style="padding:4px 8px;color:#90a4ae">{t:.1f}s</td>'
                f'<td style="padding:4px 8px;color:{st_c}">{st}</td>'
                f'</tr>'
            )
        tbl += '</table><br/>'
        lines.append(tbl)
    else:
        lines.append('<i style="color:#90a4ae">No experiments yet.</i><br/><br/>')

    # Family coverage pills
    ALL_FAMS = ['classical_ml','linear','neural_network','deep_residual',
                'ensemble_stack','attention_based','graph_neural','automl']
    lines.append('<b>Architecture Families:</b><br/>')
    for fam in ALL_FAMS:
        done = fam in fams
        color = '#2e7d32' if done else '#37474f'
        icon  = '&#10003;' if done else '&#8226;'
        lines.append(
            f'<span style="display:inline-block;background:{color};'
            f'color:#e0e0e0;border-radius:4px;padding:2px 8px;margin:2px;font-size:0.8em">'
            f'{icon} {fam}</span>'
        )
    lines.append('<br/><br/>')

    with out_widget:
        clear_output(wait=True)
        display(HTML(
            f'<div style="background:#0d1117;color:#e0e0e0;padding:16px;'
            f'border-radius:8px;font-family:monospace">'
            + ''.join(lines) + '</div>'
        ))

# ── Widget layout ────────────────────────────────────────────────────────────────────
out = widgets.Output()
_stop_flag = [False]

def _loop():
    while not _stop_flag[0]:
        _render_dashboard(out)
        for _ in range(_REFRESH_S * 2):
            if _stop_flag[0]:
                break
            time.sleep(0.5)

btn_stop = widgets.Button(description='Stop Dashboard', button_style='danger', icon='stop')
btn_refresh = widgets.Button(description='Refresh Now', button_style='info', icon='refresh')

def _on_stop(_):
    _stop_flag[0] = True
    btn_stop.description = 'Stopped'
    btn_stop.disabled = True

def _on_refresh(_):
    _render_dashboard(out)

btn_stop.on_click(_on_stop)
btn_refresh.on_click(_on_refresh)

display(widgets.HBox([btn_refresh, btn_stop]))
display(out)

# Initial render + start background refresh thread
_render_dashboard(out)
_t = threading.Thread(target=_loop, daemon=True)
_t.start()
print(f'Dashboard running (auto-refresh every {_REFRESH_S}s). Click Stop when done.')